# Post-Digitization ECG Refinement — cDDPM v2 (Kaggle, 7601)

This notebook is adapted from `v2.ipynb` to run on Kaggle with the **7601 paired noisy/clean ECG dataset**.


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
from numpy.linalg import norm

torch.manual_seed(42)
np.random.seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Running on: {DEVICE}')


In [ ]:
# Kaggle dataset auto-discovery for 7601 paired files
def pick_first(existing):
    return existing[0] if existing else None

clean_candidates = sorted(glob.glob('/kaggle/input/**/clean_new_baseline.npy', recursive=True))
noisy_candidates = sorted(glob.glob('/kaggle/input/**/noisy_new_baseline.npy', recursive=True))

if not clean_candidates or not noisy_candidates:
    # Fallback names if your Kaggle dataset uses older file names
    clean_candidates = sorted(glob.glob('/kaggle/input/**/cleannew2.npy', recursive=True))
    noisy_candidates = sorted(glob.glob('/kaggle/input/**/noisynew2.npy', recursive=True))

FILE_CLEAN = pick_first(clean_candidates)
FILE_NOISY = pick_first(noisy_candidates)

if FILE_CLEAN is None or FILE_NOISY is None:
    raise FileNotFoundError('Could not find paired clean/noisy .npy files under /kaggle/input.')

print('Using clean file:', FILE_CLEAN)
print('Using noisy file:', FILE_NOISY)

clean_raw = np.load(FILE_CLEAN).astype(np.float32)
noisy_raw = np.load(FILE_NOISY).astype(np.float32)

if clean_raw.ndim == 3:
    clean_raw = clean_raw.squeeze(1)
if noisy_raw.ndim == 3:
    noisy_raw = noisy_raw.squeeze(1)

if clean_raw.shape != noisy_raw.shape:
    raise ValueError(f'Shape mismatch: noisy {noisy_raw.shape}, clean {clean_raw.shape}')

print(f'Loaded: noisy {noisy_raw.shape}  clean {clean_raw.shape}')
print(f'NaNs   — noisy: {np.isnan(noisy_raw).sum()}  clean: {np.isnan(clean_raw).sum()}')

cosines = np.array([
    np.dot(noisy_raw[i], clean_raw[i]) / (norm(noisy_raw[i]) * norm(clean_raw[i]) + 1e-8)
    for i in range(len(noisy_raw))
])
baseline_mse = np.mean((noisy_raw - clean_raw) ** 2)
print(f'Cosine similarity — mean: {cosines.mean():.4f}  std: {cosines.std():.4f}  min: {cosines.min():.4f}')
print(f'Baseline MSE (input vs clean): {baseline_mse:.5f}')


In [ ]:
N = len(noisy_raw)
N_TRAIN = int(0.8 * N)
N_VAL = N - N_TRAIN

noisy_t = torch.FloatTensor(noisy_raw).unsqueeze(1)
clean_t = torch.FloatTensor(clean_raw).unsqueeze(1)
L = noisy_t.shape[-1]

dataset = TensorDataset(noisy_t, clean_t)
train_set, val_set = random_split(dataset, [N_TRAIN, N_VAL], generator=torch.Generator().manual_seed(42))

BATCH_SIZE = 64
NUM_WORKERS = 2
PIN_MEMORY = (DEVICE == 'cuda')

train_loader = DataLoader(
    train_set,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)
val_loader = DataLoader(
    val_set,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
)

print(f'Train: {N_TRAIN} samples | Val: {N_VAL} samples | Length: {L}')
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')


In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        device = t.device
        half_dim = self.dim // 2
        emb = np.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = t[:, None] * emb[None, :]
        return torch.cat((emb.sin(), emb.cos()), dim=-1)

class ResBlock1D(nn.Module):
    def __init__(self, channels, t_channels):
        super().__init__()
        self.norm1 = nn.GroupNorm(8, channels)
        self.conv1 = nn.Conv1d(channels, channels, 3, padding=1)
        self.t_proj = nn.Linear(t_channels, channels)
        self.norm2 = nn.GroupNorm(8, channels)
        self.conv2 = nn.Conv1d(channels, channels, 3, padding=1)
        self.act = nn.SiLU()

    def forward(self, x, t_emb):
        h = self.act(self.norm1(x))
        h = self.conv1(h)
        h = h + self.t_proj(t_emb).unsqueeze(-1)
        h = self.act(self.norm2(h))
        h = self.conv2(h)
        return x + h

class DiffusionUNet1D(nn.Module):
    def __init__(self, base_ch=128, t_dim=128):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(64),
            nn.Linear(64, t_dim),
            nn.SiLU(),
            nn.Linear(t_dim, t_dim),
        )
        self.enc_in = nn.Conv1d(2, base_ch, 3, padding=1)
        self.enc1 = ResBlock1D(base_ch, t_dim)
        self.down1 = nn.Conv1d(base_ch, base_ch * 2, 3, stride=2, padding=1)
        self.enc2 = ResBlock1D(base_ch * 2, t_dim)
        self.down2 = nn.Conv1d(base_ch * 2, base_ch * 4, 3, stride=2, padding=1)
        self.enc3 = ResBlock1D(base_ch * 4, t_dim)
        self.mid1 = ResBlock1D(base_ch * 4, t_dim)
        self.mid2 = ResBlock1D(base_ch * 4, t_dim)
        self.up2 = nn.Upsample(scale_factor=2, mode='linear', align_corners=False)
        self.dec3 = nn.Conv1d(base_ch * 4 + base_ch * 2, base_ch * 2, 3, padding=1)
        self.res3 = ResBlock1D(base_ch * 2, t_dim)
        self.up1 = nn.Upsample(scale_factor=2, mode='linear', align_corners=False)
        self.dec2 = nn.Conv1d(base_ch * 2 + base_ch, base_ch, 3, padding=1)
        self.res2 = ResBlock1D(base_ch, t_dim)
        self.out = nn.Conv1d(base_ch, 1, 1)

    def forward(self, x, cond, t):
        t_emb = self.time_mlp(t)
        h = torch.cat([x, cond], dim=1)
        e1 = self.enc1(self.enc_in(h), t_emb)
        e2 = self.enc2(self.down1(e1), t_emb)
        e3 = self.enc3(self.down2(e2), t_emb)
        m = self.mid2(self.mid1(e3, t_emb), t_emb)
        d3 = self.up2(m)
        if d3.size(2) != e2.size(2):
            d3 = d3[:, :, :e2.size(2)]
        d3 = self.res3(self.dec3(torch.cat([d3, e2], 1)), t_emb)
        d2 = self.up1(d3)
        if d2.size(2) != e1.size(2):
            d2 = d2[:, :, :e1.size(2)]
        d2 = self.res2(self.dec2(torch.cat([d2, e1], 1)), t_emb)
        return self.out(d2)

model = DiffusionUNet1D(base_ch=64, t_dim=128).to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f'Model ready. Parameters: {total_params:,}')

with torch.no_grad():
    _x = torch.zeros(2, 1, L).to(DEVICE)
    _t = torch.zeros(2, dtype=torch.long).to(DEVICE)
    _out = model(_x, _x, _t)
    print(f'Output shape check: {_out.shape}')


In [ ]:
T = 100
betas = torch.linspace(0.0001, 0.02, T).to(DEVICE)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = torch.cat([torch.tensor([1.0]).to(DEVICE), alphas_cumprod[:-1]])
sqrt_acp = torch.sqrt(alphas_cumprod)
sqrt_one_minus_acp = torch.sqrt(1.0 - alphas_cumprod)

def gather(arr, t, x_shape):
    out = arr.gather(-1, t)
    return out.reshape(t.shape[0], *((1,) * (len(x_shape) - 1)))

print(f'Schedule: T={T}, beta [{betas[0]:.5f} -> {betas[-1]:.4f}]')


In [ ]:
EPOCHS = 30
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)
criterion = nn.MSELoss()
use_amp = (DEVICE == 'cuda')
scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

train_losses, val_losses = [], []
best_val_loss = float('inf')

print(f'Training: {EPOCHS} epochs | {N_TRAIN} train / {N_VAL} val samples')

for epoch in range(EPOCHS):
    model.train()
    t_loss = 0.0
    for batch_idx, (bn, cv) in enumerate(train_loader):
        bn = bn.to(DEVICE, non_blocking=True)
        cv = cv.to(DEVICE, non_blocking=True)
        t = torch.randint(0, T, (cv.shape[0],), device=DEVICE).long()
        eps = torch.randn_like(cv)
        x_t = gather(sqrt_acp, t, cv.shape) * cv + gather(sqrt_one_minus_acp, t, cv.shape) * eps
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=use_amp):
            loss = criterion(model(x_t, bn, t), eps)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        t_loss += loss.item()
        if batch_idx == 0:
            print(f'Epoch {epoch + 1:>3}/{EPOCHS} started...')

    model.eval()
    v_loss = 0.0
    with torch.no_grad():
        for bn, cv in val_loader:
            bn = bn.to(DEVICE, non_blocking=True)
            cv = cv.to(DEVICE, non_blocking=True)
            t = torch.randint(0, T, (cv.shape[0],), device=DEVICE).long()
            eps = torch.randn_like(cv)
            x_t = gather(sqrt_acp, t, cv.shape) * cv + gather(sqrt_one_minus_acp, t, cv.shape) * eps
            with torch.amp.autocast('cuda', enabled=use_amp):
                v_loss += criterion(model(x_t, bn, t), eps).item()

    avg_t = t_loss / len(train_loader)
    avg_v = v_loss / len(val_loader)
    train_losses.append(avg_t)
    val_losses.append(avg_v)
    scheduler.step()

    if avg_v < best_val_loss:
        best_val_loss = avg_v
        torch.save(model.state_dict(), 'dm_v2_7601_best.pt')

    print(f'Epoch {epoch + 1:>3}/{EPOCHS} | train: {avg_t:.5f} | val: {avg_v:.5f} | best: {best_val_loss:.5f}')

model.load_state_dict(torch.load('dm_v2_7601_best.pt', map_location=DEVICE))
print(f'Training done. Best val loss: {best_val_loss:.5f}')

plt.figure(figsize=(9, 3))
plt.plot(train_losses, label='Train loss', color='steelblue')
plt.plot(val_losses, label='Val loss', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Noise MSE Loss')
plt.title('Training Curve - cDDPM v2 (7601 dataset)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('fig_train_curve_v2_7601.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
@torch.no_grad()
def refine(model, noisy_cond, t_start=30):
    model.eval()
    cond = noisy_cond.to(DEVICE)
    t_b = torch.full((cond.shape[0],), t_start, device=DEVICE, dtype=torch.long)
    img = gather(sqrt_acp, t_b, cond.shape) * cond + gather(sqrt_one_minus_acp, t_b, cond.shape) * torch.randn_like(cond)

    for i in range(t_start, -1, -1):
        t = torch.full((cond.shape[0],), i, device=DEVICE, dtype=torch.long)
        beta_t = gather(betas, t, img.shape)
        sqrt_1m = gather(sqrt_one_minus_acp, t, img.shape)
        sqrt_a = gather(sqrt_acp, t, img.shape)
        acp_t = gather(alphas_cumprod, t, img.shape)
        acp_prev = gather(alphas_cumprod_prev, t, img.shape)
        alpha_t = gather(alphas, t, img.shape)

        eps_pred = model(img, cond, t)
        x0_pred = (img - sqrt_1m * eps_pred) / (sqrt_a + 1e-8)
        x0_pred = torch.clamp(x0_pred, -1.5, 1.5)

        coef1 = beta_t * torch.sqrt(acp_prev) / (1.0 - acp_t + 1e-8)
        coef2 = (1.0 - acp_prev) * torch.sqrt(alpha_t) / (1.0 - acp_t + 1e-8)
        mean = coef1 * x0_pred + coef2 * img
        img = mean if i == 0 else mean + 0.5 * torch.sqrt(beta_t) * torch.randn_like(img)

    return img

def metrics(sig, ref):
    s = sig.flatten().astype(np.float64)
    r = ref.flatten().astype(np.float64)
    eps = 1e-10
    mse = np.mean((r - s) ** 2)
    mae = np.mean(np.abs(r - s))
    sp = np.sum(r ** 2)
    np_ = np.sum((r - s) ** 2)
    snr = 10 * np.log10(sp / (np_ + eps)) if sp > eps else 0.0
    prd = np.sqrt(np_ / (sp + eps)) * 100.0
    n1, n2 = norm(r), norm(s)
    cos = np.dot(r, s) / (n1 * n2 + eps) if (n1 > eps and n2 > eps) else 0.0
    return mse, mae, snr, prd, cos

print('Sampler + metrics ready')


In [ ]:
REFINE_STEPS = 20
EVAL_MAX_SAMPLES = 512  # set to None to evaluate all samples
keys = ['mse', 'mae', 'snr', 'prd', 'cos']
before = {k: [] for k in keys}
after = {k: [] for k in keys}
eval_n = len(noisy_raw) if EVAL_MAX_SAMPLES is None else min(EVAL_MAX_SAMPLES, len(noisy_raw))

print(f'Evaluating {eval_n} samples (t_start={REFINE_STEPS})...')

for i in range(eval_n):
    cond = torch.tensor(noisy_raw[i]).unsqueeze(0).unsqueeze(0)
    refined = refine(model, cond, t_start=REFINE_STEPS)
    n_sig = noisy_raw[i]
    c_sig = clean_raw[i]
    r_sig = refined.cpu().numpy().flatten()
    mb = metrics(n_sig, c_sig)
    ma = metrics(r_sig, c_sig)
    for j, k in enumerate(keys):
        before[k].append(mb[j])
        after[k].append(ma[j])

def imp(b, a, higher_is_better=False):
    if higher_is_better:
        return (a - b) / (abs(b) + 1e-10) * 100
    return (b - a) / (abs(b) + 1e-10) * 100

db = {k: np.mean(v) for k, v in before.items()}
da = {k: np.mean(v) for k, v in after.items()}

results = pd.DataFrame({
    'Metric': ['MSE', 'MAE', 'SNR (dB)', 'PRD (%)', 'Cosine Sim'],
    'Input': [db['mse'], db['mae'], db['snr'], db['prd'], db['cos']],
    'Refined': [da['mse'], da['mae'], da['snr'], da['prd'], da['cos']],
    'Improvement (%)': [
        imp(db['mse'], da['mse']),
        imp(db['mae'], da['mae']),
        imp(db['snr'], da['snr'], higher_is_better=True),
        imp(db['prd'], da['prd']),
        imp(db['cos'], da['cos'], higher_is_better=True),
    ]
})

print(results.to_string(index=False, formatters={
    'Input': '{:.4f}'.format,
    'Refined': '{:.4f}'.format,
    'Improvement (%)': '{:+.2f}%'.format
}))

mse_imp = [(b - a) / (abs(b) + 1e-10) * 100 for b, a in zip(before['mse'], after['mse'])]
improved = sum(1 for x in mse_imp if x > 0)
print(f'Samples improved: {improved}/{len(mse_imp)}')
print(f'Best improvement: {max(mse_imp):+.2f}%')
print(f'Worst case: {min(mse_imp):+.2f}%')

results.to_csv('results_v2_7601.csv', index=False)
print('Saved: results_v2_7601.csv')
